## Agent RAG

In [46]:
import os
import re
import requests
from dotenv import load_dotenv
from typing import Dict, List, Tuple, Optional
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import PyMuPDFLoader
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore
from langchain_core.retrievers import BaseRetriever
from langchain_core.tools import create_retriever_tool
from langgraph.prebuilt import create_react_agent
from langsmith import Client
from langchain_core.prompts import PromptTemplate

In [7]:
load_dotenv()

GEMMA_MODEL = os.getenv("GEMMA_MODEL")
BASE_URL = os.getenv("BASE_URL")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")
API_KEY = os.getenv("API_KEY")
QDRANT_URL = os.getenv("QDRANT_URL")

In [55]:
llm = ChatOpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
    model=GEMMA_MODEL,
)

In [4]:
urls = [
   "https://raw.githubusercontent.com/llama-index-tutorial/llama-index-tutorial/main/ch06/ict_japan_2024.pdf",
   "https://raw.githubusercontent.com/llama-index-tutorial/llama-index-tutorial/main/ch06/ict_usa_2024.pdf"
]


In [5]:
for url in urls:
    filename = url.split("/")[-1]
    response = requests.get(url)

    with open(filename, "wb") as f:
        f.write(response.content)
    print(f"{filename} downloaded")

ict_japan_2024.pdf downloaded
ict_usa_2024.pdf downloaded


In [8]:
embed = OpenAIEmbeddings(
    base_url=BASE_URL,
    api_key=API_KEY,
    model=EMBEDDING_MODEL,
    check_embedding_ctx_length=False,
)

In [11]:
qdrant_client = QdrantClient(
    url=QDRANT_URL,
)

vector_size = len(embed.embed_query("차원 확인"))
print(vector_size)
qdrant_client.create_collection(
    collection_name="react_agent",
    vectors_config=VectorParams(
        size=vector_size,
        distance=Distance.COSINE,
    ),
)

vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name="react_agent",
    embedding=embed,
)


1024


In [22]:
def create_pdf_retriever(
        pdf_path: str,
        collection_name: str,
        chunk_size: int = 512,
        chunk_overlap: int = 0,
)-> BaseRetriever:

    loader = PyMuPDFLoader(pdf_path)
    data = loader.load()

    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    doc_splits = text_splitter.split_documents(data)

    qdrant_client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(
        size=vector_size,
        distance=Distance.COSINE,
        ),
    )

    vectorstore = QdrantVectorStore(
        client=qdrant_client,
        collection_name=collection_name,
        embedding=embed,
    )

    vectorstore.add_documents(doc_splits)

    return vectorstore.as_retriever()



In [23]:
retriever_japan = create_pdf_retriever(
    pdf_path="ict_japan_2024.pdf",
    collection_name="japan_ict",
)

retriever_usa = create_pdf_retriever(
    pdf_path="ict_usa_2024.pdf",
    collection_name="usa_ict",
)

In [32]:
jp_engine = create_retriever_tool(
    retriever=retriever_japan,
    name = "japan_ict",
    description="일본의 ICT 시장 동향 정보를 제공합니다. 일본 ICT와 관련된 질문은 해당 도구를 사용하세요",
)

usa_engine = create_retriever_tool(
    retriever=retriever_usa,
    name = "usa_ict",
    description="미국의 ICT 시장 동향 정보를 제공합니다. 미국 ICT와 관련된 질문은 해당 도구를 사용하세요",
)

In [37]:
tools = [jp_engine, usa_engine]
tool_map: Dict[str, object] = {t.name: t for t in tools}

In [39]:
def _format_tools_for_prompt(ts: List[object]) -> Tuple[str, str]:
    lines, names = [], []
    for t in ts:
        names.append(t.name)
        desc = getattr(t, "description", "")
        lines.append(f"{t.name}: {desc}")
    return "\n".join(lines), ", ".join(names)

def _render_prompt(user_input: str, scratchpad: str) -> str:
    tools_str, tool_names = _format_tools_for_prompt(tools)
    return prompt.format(
        tools=tools_str,
        tool_names=tool_names,
        input=user_input,
        agent_scratchpad=scratchpad,
    )


In [51]:

ACTION_RE = re.compile(
    r"^Action\s*:\s*(?P<tool>.+?)\s*$",
    re.MULTILINE,
)

ACTION_INPUT_RE = re.compile(
    r"^Action Input\s*:\s*(?P<input>.+?)\s*$",
    re.MULTILINE,
)

FINAL_ANSWER_RE = re.compile(
    r"^Final Answer\s*:\s*(?P<final>[\s\S]+)$",
    re.IGNORECASE | re.MULTILINE,
)

In [53]:
from pprint import pp

react_template = '''다음 질문에 최선을 다해 답변하세요. 당신은 다음 도구들에 접근할 수 있습니다:

{tools}

다음 형식을 사용하세요:

Question: 답변해야 하는 입력 질문
Thought: 무엇을 할지 항상 생각하세요
Action: 취해야 할 행동, [{tool_names}] 중 하나여야 합니다. 리스트에 있는 도구 중 1개를 택하십시오.
Action Input: 행동에 대한 입력값
Observation: 행동의 결과
... (이 Thought/Action/Action Input/Observation의 과정이 N번 반복될 수 있습니다)
Thought: 이제 최종 답변을 알겠습니다
Final Answer: 원래 입력된 질문에 대한 최종 답변 (한글로 작성하십시오.)

## 추가적인 주의사항
- 반드시 [Thought -> Action -> Action Input -> Observation] 순서를 준수하십시오. 항상 Action 전에는 Thought가 먼저 나와야 합니다.
- 최종 답변에는 최대한 많은 내용을 포함하십시오.
- 한 번의 검색으로 해결되지 않을 것 같다면 문제를 분할하여 푸는 것도 고려하십시오.
- 정보가 취합되었다면 불필요하게 사이클을 반복하지 마십시오.
- 묻지 않은 정보를 찾으려고 도구를 사용하지 마십시오.

시작하세요!

Question: {input}
{agent_scratchpad}'''

prompt = PromptTemplate.from_template(react_template)

In [52]:
def _parse_action_and_input(text: str) -> Tuple[Optional[str], Optional[str]]:
    m_final = FINAL_ANSWER_RE.search(text)
    if m_final:
        return "__FINAL__", m_final.group("final").strip()
    m_act = ACTION_RE.search(text)
    m_in = ACTION_INPUT_RE.search(text)
    if m_act and m_in:
        return m_act.group("tool").strip(), m_in.group("input").strip()
    return None, None


In [48]:
def _observation_to_text(observation_obj) -> str:
    if isinstance(observation_obj, list):
        def doc_to_str(d):
            try:
                meta = getattr(d, "metadata", {}) or {}
                src = meta.get("source") or meta.get("file_path") or ""
                txt = getattr(d, "page_content", "")
                if len(txt) > 500:
                    txt = txt[:500] + "..."
                return f"[source={src}] {txt}"
            except Exception:
                return str(d)
        return "\n".join(doc_to_str(d) for d in observation_obj[:5])
    return str(observation_obj)

In [49]:
def run_react(user_input: str, max_iters: int = 8) -> Dict[str, str]:
    scratchpad = ""
    for _ in range(max_iters):
        rendered = _render_prompt(user_input, scratchpad)
        resp = llm.invoke(rendered)
        text = resp.content if hasattr(resp, "content") else str(resp)

        tool, action_input = _parse_action_and_input(text)
        if tool is None:
            hint = "\n[파싱안내] 형식을 엄격히 따르세요. 반드시 'Action:'와 'Action Input:'을 한 줄씩 제공하십시오.\n"
            scratchpad += f"{text}\n{hint}"
            continue

        if tool == "__FINAL__":
            final_answer = action_input
            return {"output": final_answer, "log": scratchpad + "\n" + text}

        if tool not in tool_map:
            observation = f"[에러] 존재하지 않는 도구입니다: {tool}"
            scratchpad += f"{text}\nObservation: {observation}\n"
            continue

        try:
            observation_obj = tool_map[tool].invoke(action_input)
            observation = _observation_to_text(observation_obj)
            scratchpad += f"{text}\nObservation: {observation}\n"
        except Exception as e:
            scratchpad += f"{text}\nObservation: [도구실행오류] {e}\n"

    return {
        "output": "반복 한도를 초과했습니다. 질문을 더 구체화해 주세요.",
        "log": scratchpad,
    }

In [56]:
result = run_react("한국과 미국의 ICT 기관 협력 사례")
print("최종 답변:", result["output"])
print("\n=== 실행 로그 ===\n")
print(result["log"])

최종 답변: 한국과 미국의 ICT 기관 간 협력 사례는 양국 간의 제도적, 외교적 교류를 바탕으로 이루어지고 있습니다.

**주요 협력 배경 및 제도적 기반:**

1.  **한-미 FTA의 역할:** 2012년 한-미 FTA가 발효되었고, 이후 2019년 개정 의정서까지 발효되면서 양국 간의 무역과 투자의 법적 틀이 더욱 강화되었습니다. 이는 기업 활동 및 협력에 중요한 기반을 제공하고 있습니다.
2.  **정부 간 고위급 교류:** 양국 정부 및 의회 대표단이 워싱턴 D.C., 뉴욕, 미시간 등지를 방문하며 긴밀한 관계를 유지하고 있으며, 이는 ICT 분야의 협력을 위한 고수준의 대화가 지속되고 있음을 보여줍니다.
3.  **ICT 전문 기관 간의 협력:** 특히 한국의 과학기술정보통신부(과기정통부)와 미국 기관들 사이에서 ICT 분야의 구체적인 협력 사례들이 활발하게 논의되고 주목받고 있습니다.

**협력 분야 및 동향:**

*   자료에는 양국 ICT의 발전 동향으로 인공지능(AI) 챗봇 개발, 양자 컴퓨팅, 우주 클라우드 컴퓨팅 등 첨단 기술 분야가 언급되어 있으며, 이러한 첨단 기술 이슈들은 향후 양국 ICT 기관들이 협력할 수 있는 주요 주제군을 형성하고 있습니다.

요약하자면, 양국은 FTA라는 견고한 무역 환경 위에서 시작하여, 각 정부 기관 및 전문 부처 간의 고위급 대화와 기술 교류를 통해 ICT 협력 관계를 구축하고 있음을 확인할 수 있습니다.

=== 실행 로그 ===

Thought: 사용자는 한국과 미국의 ICT 기관 협력 사례를 요청했습니다. 제가 사용할 수 있는 도구는 `japan_ict`과 `usa_ict`입니다. 미국과의 협력 사례를 찾으려면 `usa_ict` 도구를 사용해야 합니다. 해당 도구에 한국과의 협력 사례에 대한 정보를 요청하겠습니다.
Action: usa_ict

[파싱안내] 형식을 엄격히 따르세요. 반드시 'Action:'와 'Action Input:'을 한 줄씩 제공하십시오.
Question: 한국과 미국의 